In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Nháp

In [ ]:
import cv2
import numpy as np

pred_path = "/content/drive/MyDrive/KLTN/my_dataset/data1/segmentation_masks/000001.png"

mask = cv2.imread(
    pred_path,
    cv2.IMREAD_UNCHANGED
)

print("shape:", mask.shape)
print("dtype:", mask.dtype)

if len(mask.shape) == 2:
    print("Single-channel mask")
    print("Unique IDs:")
    print(np.unique(mask)[:50])

else:
    print("RGB image")
    colors = np.unique(
        mask.reshape(-1,3),
        axis=0
    )
    print("num colors =", len(colors))
    print(colors[:20])

shape: (1242, 2208, 3)
dtype: uint8
RGB image
num colors = 15
[[  0 161 115]
 [ 22 167 113]
 [ 34  24 121]
 [ 39 240 214]
 [ 61 179  95]
 [ 71 187  99]
 [ 72  43 167]
 [ 77 226 209]
 [105  53  59]
 [109 237 116]
 [131   3 101]
 [198 111  26]
 [243  32 241]
 [245 149  14]
 [253  79 204]]


In [ ]:
import gzip
import numpy as np

pred_path = "/content/drive/MyDrive/KLTN/my_dataset/data1/depth_maps/000001.float16.gz"

with gzip.open(pred_path, "rb") as f:
    arr = np.frombuffer(
        f.read(),
        dtype=np.float16
    )

h = int(arr[0])
w = int(arr[1])

depth = arr[2:].reshape(h,w)

print("shape:", depth.shape)

print("min :", depth.min())
print("max :", depth.max())
print("mean:", depth.mean())

shape: (1242, 2208)
min : 0.3276
max : 31.86
mean: 9.41


#### Segformer B5

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

INPUT_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/left/video_frames"
OUTPUT_DIR = "/content/drive/MyDrive/KLTN/sanpo_gen/data1/segmentation_masks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640"
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640"
)

model.to(DEVICE)
model.eval()

files = sorted(os.listdir(INPUT_DIR))

for file in tqdm(files):

    if not file.lower().endswith(
        (".png",".jpg",".jpeg")
    ):
        continue

    path = os.path.join(INPUT_DIR,file)

    image = cv2.imread(path)
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    h,w = image.shape[:2]

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k:v.to(DEVICE)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

    logits = outputs.logits

    logits = torch.nn.functional.interpolate(
        logits,
        size=(h,w),
        mode="bilinear",
        align_corners=False
    )

    pred = logits.argmax(
        dim=1
    )[0]

    pred = pred.cpu().numpy().astype(
        np.uint8
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        os.path.splitext(file)[0] + ".png"
    )

    cv2.imwrite(save_path,pred)

print("Done")

#### Depth Anything V2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/KLTN

!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/KLTN
Cloning into 'Depth-Anything-V2'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 146 (delta 30), reused 26 (delta 26), pack-reused 86 (from 2)
Receiving objects: 100% (146/146), 45.18 MiB | 18.60 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Updating files: 100% (74/74), done.


In [ ]:
import sys

sys.path.append(
    "/content/drive/MyDrive/KLTN/Depth-Anything-V2"
)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
import os

# Tải về cache của Hugging Face
model_path = hf_hub_download(
    repo_id="depth-anything/Depth-Anything-V2-Large",
    filename="depth_anything_v2_vitl.pth"
)

save_dir = "/content/drive/MyDrive/KLTN/models"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "depth_anything_v2_vitl.pth"
)

shutil.copy(model_path, save_path)

print("Saved to:", save_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


depth_anything_v2_vitl.pth:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Saved to: /content/drive/MyDrive/KLTN/models/depth_anything_v2_vitl.pth


In [ ]:
import os
import cv2
import gzip
import torch
import numpy as np
from tqdm import tqdm

from depth_anything_v2.dpt import DepthAnythingV2

INPUT_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/left/video_frames"
OUTPUT_DIR = "/content/drive/MyDrive/KLTN/sanpo_gen/data1/depth_maps"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===== Create Model =====

model = DepthAnythingV2(
    encoder="vitl",
    features=256,
    out_channels=[
        256,
        512,
        1024,
        1024
    ]
)

# ===== Load Weight =====

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/KLTN/models/depth_anything_v2_vitl.pth",
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

# ===== Save Depth =====

def save_depth(depth, path):

    h, w = depth.shape

    header = np.array(
        [h, w],
        dtype=np.float16
    )

    data = np.concatenate([
        header,
        depth.flatten().astype(
            np.float16
        )
    ])

    with gzip.open(
        path,
        "wb"
    ) as f:

        f.write(
            data.tobytes()
        )

# ===== Predict =====

files = sorted(
    os.listdir(INPUT_DIR)
)

for file in tqdm(files):

    if not file.lower().endswith(
        (".png", ".jpg", ".jpeg")
    ):
        continue

    image_path = os.path.join(
        INPUT_DIR,
        file
    )

    image = cv2.imread(
        image_path
    )

    depth = model.infer_image(
        image
    )

    depth = depth.astype(
        np.float32
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        os.path.splitext(file)[0]
        +
        ".float16.gz"
    )

    save_depth(
        depth,
        save_path
    )

print("Done")

100%|██████████| 528/528 [26:17<00:00,  2.99s/it]

Done


#### Đánh giá depth

In [ ]:
import os
import cv2
import gzip
import numpy as np
from tqdm import tqdm

GT_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/left/depth_maps"
PRED_DIR = "/content/drive/MyDrive/KLTN/sanpo_gen/data1/depth_maps"


def load_depth(path):

    with gzip.open(path, "rb") as f:
        arr = np.frombuffer(
            f.read(),
            dtype=np.float16
        )

    h = int(arr[0])
    w = int(arr[1])

    depth = arr[2:].reshape(h, w)

    return depth.astype(np.float32)


mae_sum = 0.0
rmse_sum = 0.0
absrel_sum = 0.0

delta1_sum = 0.0
delta2_sum = 0.0
delta3_sum = 0.0

count = 0

files = sorted(os.listdir(GT_DIR))

for file in tqdm(files):

    if not file.endswith(".float16.gz"):
        continue

    gt_path = os.path.join(
        GT_DIR,
        file
    )

    pred_path = os.path.join(
        PRED_DIR,
        file
    )

    if not os.path.exists(pred_path):
        continue

    gt = load_depth(gt_path)
    pred = load_depth(pred_path)

    # resize prediction nếu khác kích thước

    if pred.shape != gt.shape:

        pred = cv2.resize(
            pred,
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

    # bỏ pixel invalid

    mask = (
        (gt > 0)
        &
        np.isfinite(gt)
        &
        np.isfinite(pred)
    )

    gt = gt[mask]
    pred = pred[mask]

    if len(gt) == 0:
        continue

    # scale alignment
    # rất quan trọng với Depth Anything

    scale = np.median(gt) / np.median(pred)

    pred = pred * scale

    mae = np.mean(
        np.abs(gt - pred)
    )

    rmse = np.sqrt(
        np.mean(
            (gt - pred) ** 2
        )
    )

    absrel = np.mean(
        np.abs(gt - pred)
        / gt
    )

    ratio = np.maximum(
        gt / pred,
        pred / gt
    )

    delta1 = np.mean(
        ratio < 1.25
    )

    delta2 = np.mean(
        ratio < 1.25**2
    )

    delta3 = np.mean(
        ratio < 1.25**3
    )

    mae_sum += mae
    rmse_sum += rmse
    absrel_sum += absrel

    delta1_sum += delta1
    delta2_sum += delta2
    delta3_sum += delta3

    count += 1

print("\n===== DEPTH EVALUATION =====")

print(
    f"MAE    : {mae_sum/count:.4f}"
)

print(
    f"RMSE   : {rmse_sum/count:.4f}"
)

print(
    f"AbsRel : {absrel_sum/count:.4f}"
)

print(
    f"δ1     : {delta1_sum/count:.4f}"
)

print(
    f"δ2     : {delta2_sum/count:.4f}"
)

print(
    f"δ3     : {delta3_sum/count:.4f}"
)

  0%|          | 0/528 [00:00<?, ?it/s]/tmp/ipykernel_1084/142230587.py:109: RuntimeWarning: divide by zero encountered in divide
  gt / pred,
100%|██████████| 528/528 [00:43<00:00, 12.10it/s]


===== DEPTH EVALUATION =====
MAE    : 7.9117
RMSE   : 10.0450
AbsRel : 2.0878
δ1     : 0.1596
δ2     : 0.2601
δ3     : 0.3426


In [ ]:
gt_path = "/content/drive/MyDrive/KLTN/dataset/data1/left/depth_maps/000001.float16.gz"
pred_path = "/content/drive/MyDrive/KLTN/my_dataset/data1/depth_maps/000001.float16.gz"

# Assuming load_depth function is defined in a previous cell
gt = load_depth(gt_path)
pred = load_depth(pred_path)

print("GT shape:", gt.shape)
print("Pred shape:", pred.shape)

print("GT median:", np.median(gt))
print("Pred median:", np.median(pred))

GT shape: (720, 1280)
Pred shape: (1242, 2208)
GT median: 3.4921875
Pred median: 6.216797


### code fine_tuned

In [ ]:
ROAD_IDS = [1]
SIDEWALK_IDS = [4]

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

MASK_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/segmentation_masks"
OUTPUT_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/binary_masks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

WALKABLE_IDS = [
    1,   # road
    4    # sidewalk
]

files = sorted(os.listdir(MASK_DIR))

for file in tqdm(files):

    path = os.path.join(MASK_DIR, file)

    mask = cv2.imread(
        path,
        cv2.IMREAD_UNCHANGED
    )

    binary = np.zeros_like(
        mask,
        dtype=np.uint8
    )

    binary[
        np.isin(
            mask,
            WALKABLE_IDS
        )
    ] = 1

    cv2.imwrite(
        os.path.join(
            OUTPUT_DIR,
            file
        ),
        binary
    )

print("Done")

In [ ]:
from transformers import (
    SegformerForSemanticSegmentation
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640",
    num_labels=2,
    ignore_mismatched_sizes=True
)

In [ ]:
mask = np.array(mask)

mask = mask.astype(
    np.int64
)

In [ ]:
import evaluate
import torch

metric = evaluate.load(
    "mean_iou"
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    logits = torch.tensor(logits)

    preds = torch.nn.functional.interpolate(
        logits,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    preds = preds.argmax(
        dim=1
    ).numpy()

    results = metric.compute(
        predictions=preds,
        references=labels,
        num_labels=2,
        ignore_index=255
    )

    return {
        "mIoU":
            results["mean_iou"],
        "PixelAcc":
            results["mean_accuracy"]
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir=
        "/content/drive/MyDrive/KLTN/segformer_sanpo",

    learning_rate=
        6e-5,

    per_device_train_batch_size=
        2,

    per_device_eval_batch_size=
        2,

    num_train_epochs=
        20,

    evaluation_strategy=
        "epoch",

    save_strategy=
        "epoch",

    load_best_model_at_end=
        True,

    metric_for_best_model=
        "mIoU",

    remove_unused_columns=
        False
)

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

MASK_DIR = "/content/drive/MyDrive/KLTN/dataset/data1/left/segmentation_masks"

all_ids = set()

for file in tqdm(os.listdir(MASK_DIR)):

    mask = cv2.imread(
        os.path.join(MASK_DIR,file),
        cv2.IMREAD_UNCHANGED
    )

    all_ids.update(
        np.unique(mask).tolist()
    )

print(sorted(all_ids))

100%|██████████| 528/528 [02:37<00:00,  3.35it/s]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29]
